# Team Winners
**Joy, Radiah, Ellie**

<a href="https://colab.research.google.com/github/inthecrowsnest/DeepLearningProject/blob/main/m1_updated_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 1: models and error

Predict the origin of a Hungarian lemma (native, German, Slavic, Turkic).

Input: `tokens_dataset_updated.csv` from `preprocessing.py`: 1000 labeled words from the subtitles (tokenised and lemmatised with HuSpaCy, stop words removed, labeled from `labeled_words.csv`, with subtitle counts). Everything is labeled first; this notebook cleans and only then splits.

Everything is evaluated with 5-fold stratified cross-validation and reported as mean ± std. Primary metric: macro-F1.

Models:
1. Majority class (floor)
2. Frequency only: logistic regression on log subtitle count (measures the frequency shortcut)
3. Character 2–4-gram logistic regression (strong simple baseline)
4. Character-level bidirectional GRU (main model)

In [ ]:
# In Colab, fetch the repo so the data file is available
import os
if not os.path.exists('tokens_dataset_updated.csv'):
    !git clone -q https://github.com/inthecrowsnest/DeepLearningProject.git
    %cd DeepLearningProject

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 0

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

## Data

`tokens_dataset_updated.csv` comes from `preprocessing.py`: subtitle sentences tokenised and lemmatised with HuSpaCy, stop words, punctuation and non-alphabetic tokens removed, and every lemma labeled from `labeled_words.csv` until 1000 labeled words are found. `count` is how often the lemma appeared in the subtitles read. Only the four project classes are kept (a safety check for older files).

In [ ]:
CLASSES = ['inherited', 'german', 'slavic', 'turkic']   # 'inherited' = native

tokens = pd.read_csv('tokens_dataset_updated.csv')
print('rows in file:', len(tokens))
tokens = tokens[tokens['label'].isin(CLASSES)]
print('rows in the four classes:', len(tokens))
if 'count' not in tokens.columns:
    print('WARNING: no count column (old tokens file); every word gets count 1, '
          'so the frequency baseline is meaningless. Rerun preprocessing.py.')
tokens.head()

### One row per word, then hand-picked cleaning
The cleaning rules from `clean_labeled_subtitles.py` remove pronouns, interjections, proper names and lemmatiser junk that happened to match a dictionary entry.

In [ ]:
FUNCTION = {'ő', 'ó', 'ez', 'az', 'én', 'mi', 'te', 'ti', 'ön', 'két', 'jé', 'al', 'ál'}
NAMES = {'müller', 'tóth', 'gyula', 'ada', 'semmelweis', 'beethoven', 'konrád', 'léna',
         'moszkva', 'visegrád', 'frankfurt', 'kazahsztán'}
JUNK = {'várta', 'sing', 'os', 'im', 'ör', 'basz', 'bű', 'eng', 'alt'}

if 'count' in tokens.columns:      # one row per word with its subtitle count
    df = tokens.drop_duplicates('lemma')[['lemma', 'count', 'label']]
else:                              # one row per token occurrence: count them
    df = tokens.groupby('lemma').agg(count=('lemma', 'size'), label=('label', 'first')).reset_index()
df = df.sort_values('count', ascending=False, ignore_index=True)

before = len(df)
df = df[~df['lemma'].isin(FUNCTION | NAMES | JUNK)].reset_index(drop=True)
print(f'{before} labeled words -> {len(df)} after cleaning')

df['y'] = df['label'].map({c: i for i, c in enumerate(CLASSES)})
counts = df['label'].value_counts().reindex(CLASSES)
print(pd.DataFrame({'words': counts, 'share': (counts / len(df)).round(3)}))
print('total', len(df))

In [ ]:
# Five real rows: one per class plus one more
pd.concat([df.groupby('label').sample(1, random_state=SEED),
           df.sample(1, random_state=SEED + 1)])[['lemma', 'count', 'label']]

In [ ]:
# Median subtitle count per class: native words are far more frequent
df.groupby('label')['count'].median().reindex(CLASSES)

## Folds

5 stratified folds by word. Inside each training fold, 15% of words are held out as validation for early stopping (GRU only; the baselines train on the full training fold).

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = []
for train_idx, test_idx in skf.split(df, df['y']):
    tr_idx, val_idx = train_test_split(train_idx, test_size=0.15,
                                       stratify=df['y'].iloc[train_idx], random_state=SEED)
    folds.append((train_idx, tr_idx, val_idx, test_idx))

for k, (train_idx, tr_idx, val_idx, test_idx) in enumerate(folds):
    print(f'fold {k}: train {len(tr_idx)}  val {len(val_idx)}  test {len(test_idx)}')

In [ ]:
def scores(y_true, y_pred):
    rec = recall_score(y_true, y_pred, labels=range(len(CLASSES)), average=None, zero_division=0)
    out = {'accuracy': accuracy_score(y_true, y_pred),
           'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)}
    out.update({f'{c}_recall': r for c, r in zip(CLASSES, rec)})
    return out

results = {}       # model name -> list of per-fold score dicts
oof_preds = {}     # model name -> out-of-fold predictions for every word

## Baselines

In [ ]:
def run_sklearn(name, make_model, make_features):
    results[name], preds = [], np.zeros(len(df), dtype=int)
    for train_idx, _, _, test_idx in folds:
        X_train, X_test = make_features(df.iloc[train_idx], df.iloc[test_idx])
        model = make_model().fit(X_train, df['y'].iloc[train_idx])
        preds[test_idx] = model.predict(X_test)
        results[name].append(scores(df['y'].iloc[test_idx], preds[test_idx]))
    oof_preds[name] = preds

def log_count(train, test):
    return np.log(train[['count']].values), np.log(test[['count']].values)

def char_ngrams(train, test):
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
    return vec.fit_transform(train['lemma']), vec.transform(test['lemma'])

run_sklearn('Majority baseline', lambda: DummyClassifier(strategy='most_frequent'), log_count)
run_sklearn('Frequency only', lambda: LogisticRegression(class_weight='balanced'), log_count)
run_sklearn('Char n-gram LR', lambda: LogisticRegression(class_weight='balanced', max_iter=2000), char_ngrams)

for name in results:
    print(f"{name:20s} macro-F1 {np.mean([r['macro_f1'] for r in results[name]]):.3f}")

## Character-level GRU

Input: the word as characters with start and end markers. Embedding (16) → 1-layer bidirectional GRU (32 per direction) → dropout → linear to 4 classes.

Loss: cross-entropy weighted by inverse class frequency in the training fold. Early stopping keeps the epoch with the best validation macro-F1.

In [ ]:
PAD, BOS, EOS = 0, 1, 2
chars = sorted(set(''.join(df['lemma'])))
char_to_id = {ch: i + 3 for i, ch in enumerate(chars)}
VOCAB_SIZE = len(char_to_id) + 3
print('characters', len(chars), '| vocabulary with pad/start/end', VOCAB_SIZE)

def encode(word):
    return torch.tensor([BOS] + [char_to_id[ch] for ch in word] + [EOS])

def make_batch(idx):
    seqs = [encode(w) for w in df['lemma'].iloc[idx]]
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), lengths, torch.tensor(df['y'].iloc[idx].values)

In [ ]:
# Hyperparameters
EMB_DIM = 16
HIDDEN = 32
DROPOUT = 0.3
LR = 3e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 15   # epochs without validation macro-F1 improvement before stopping

class CharGRU(nn.Module):
    def __init__(self, dropout=DROPOUT):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMB_DIM, padding_idx=PAD)
        self.gru = nn.GRU(EMB_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * HIDDEN, len(CLASSES))

    def forward(self, x, lengths):
        packed = pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)                      # h: (2, batch, HIDDEN)
        return self.out(self.drop(torch.cat([h[0], h[1]], dim=1)))

model = CharGRU()
print('trainable parameters', sum(p.numel() for p in model.parameters() if p.requires_grad))
for name, p in model.named_parameters():
    print(f'  {name:22s} {p.numel()}')

In [ ]:
def class_weights(idx):
    freq = np.bincount(df['y'].iloc[idx], minlength=len(CLASSES))
    return torch.tensor(len(idx) / (len(CLASSES) * freq), dtype=torch.float)

def predict(model, idx):
    model.eval()
    x, lengths, y = make_batch(idx)
    with torch.no_grad():
        logits = model(x, lengths)
    return logits, y

def train_gru(tr_idx, val_idx):
    set_seed()
    model = CharGRU()
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(tr_idx))
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
    best_f1, best_state, since_best = -1, None, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        order = np.random.permutation(tr_idx)
        total = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            x, lengths, y = make_batch(order[start:start + BATCH_SIZE])
            loss = loss_fn(model(x, lengths), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
        history['train_loss'].append(total / len(tr_idx))

        logits, y = predict(model, val_idx)
        history['val_loss'].append(loss_fn(logits, y).item())
        val_f1 = f1_score(y, logits.argmax(1), average='macro')
        history['val_macro_f1'].append(val_f1)

        if val_f1 > best_f1:
            best_f1, best_state, since_best = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            since_best += 1
            if since_best >= PATIENCE:
                break

    model.load_state_dict(best_state)
    history['best_epoch'] = int(np.argmax(history['val_macro_f1']))
    return model, history

### Twenty-word check

With dropout off and no early stopping, the model should drive training loss on 20 words to near zero. If it cannot, something is broken.

In [ ]:
set_seed()
tiny = df.groupby('label').sample(5, random_state=SEED).index.values
tiny_model = CharGRU(dropout=0.0)
opt = torch.optim.Adam(tiny_model.parameters(), lr=1e-2)
x, lengths, y = make_batch(tiny)
for step in range(300):
    loss = nn.functional.cross_entropy(tiny_model(x, lengths), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f'final loss {loss.item():.4f} | train accuracy {(tiny_model(x, lengths).argmax(1) == y).float().mean():.2f}')

### Cross-validation

In [ ]:
results['Char-GRU'], histories = [], []
oof_preds['Char-GRU'] = np.zeros(len(df), dtype=int)
for k, (_, tr_idx, val_idx, test_idx) in enumerate(folds):
    model, history = train_gru(tr_idx, val_idx)
    histories.append(history)
    logits, y = predict(model, test_idx)
    oof_preds['Char-GRU'][test_idx] = logits.argmax(1).numpy()
    results['Char-GRU'].append(scores(y, logits.argmax(1)))
    print(f"fold {k}: best epoch {history['best_epoch'] + 1}  test macro-F1 {results['Char-GRU'][-1]['macro_f1']:.3f}")

## Results (mean ± std over 5 folds)

In [ ]:
cols = ['accuracy', 'macro_f1'] + [f'{c}_recall' for c in CLASSES]
table = pd.DataFrame({
    name: {c: f"{np.mean([r[c] for r in runs]):.3f} ± {np.std([r[c] for r in runs]):.3f}" for c in cols}
    for name, runs in results.items()
}).T
table

In [ ]:
h = histories[0]
epochs = range(1, len(h['train_loss']) + 1)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(epochs, h['train_loss'], label='train loss')
ax1.plot(epochs, h['val_loss'], label='validation loss')
ax1.set_xlabel('epoch')
ax1.set_ylabel('weighted cross-entropy')
ax2 = ax1.twinx()
ax2.plot(epochs, h['val_macro_f1'], color='tab:green', linestyle='--', label='validation macro-F1')
ax2.set_ylabel('macro-F1')
ax1.axvline(h['best_epoch'] + 1, color='grey', linestyle=':', label='kept epoch')
fig.legend(loc='lower center', ncol=4, frameon=False)
ax1.set_title('Char-GRU training curve (fold 0)')
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
names = ['Native', 'German', 'Slavic', 'Turkic']
for ax, name in zip(axes, ['Char n-gram LR', 'Char-GRU']):
    cm = confusion_matrix(df['y'], oof_preds[name], normalize='true')
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(4), names)
    ax.set_yticks(range(4), names)
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    ax.set_title(f'{name} (row-normalised, all folds)')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                    color='white' if cm[i, j] > 0.5 else 'black')
plt.tight_layout()
plt.show()